# v123_tight_rule — v122 decoded for costlier false merges

| Field | Value |
|---|---|
| **Version** | `v123_tight_rule` |
| **Plan group** | E2 (threshold optimisation: the decision rule alone is re-tuned) |
| **Parent version** | `v122` (`v122_full_stack`: mock F0.5 0.98306, est_public 0.97466; public 0.966, assumed: see §1) |
| **Author** | M3 |
| **Date** | 2026-09-27 |
| **Status** | INVESTIGATE: judged by the public upload |
| **Hypothesis** | v122's own probabilities, decoded by a rule tuned for false merges that cost 3× a missed match (v122: 1.45×), raise the public score although they lower mock F0.5 and est_public: the gains that reached the leaderboard were precision gains, recall gains did not, and the test is harder than the mock. |

The tight mock stopped tracking the leaderboard after v107. It predicted v107's public score to
within 0.0001 (est_public 0.9659, public 0.966), then rose to 0.9731 for v110 and 0.9747 for
v122 while the public score stayed at 0.966. The changes that did move the public score bought
precision: the rule re-tuned on the mock (v103, +0.006) and the two-stage matcher with its tight
rule (v107, +0.005). v110 and v122 gained mostly recall on the mock and nothing on the
leaderboard. v122's own probabilities also say the test is harder than the mock (§6), most of
all in France, which train lacks. So a false merge probably costs more on test than the ×1.45
fitted on v101 and v103 (`mock.py`).

v123 changes only that weight. No model is retrained: v122's saved stage-2 probabilities of the
mock and of the test are decoded with rules tuned for false-merge weights 1.45 (v122's), 2, 3, 4
and 6. Mock F0.5 and est_public are expected to fall as the weight rises, since the rule is then
tuned for another score than the ones they measure, so the mock cannot judge v123: the public
upload of weight 3 does.

The experiment is the script `run_v123.py` in this folder, not this notebook: it reads v122's
artifacts, sweeps the weights, writes the test files and ends with `tracking.log_result`
(`metrics.json` here, the `v123` row of `experiments/experiments.csv`). **This notebook
recomputes nothing**: its cells read `metrics.json`, v122's `metrics.json` and `run.log`, and
print "run not finished" until the script has logged its result.

```
v122's saved artifacts (no model trained)
    mock_scored.parquet   stage-2 probabilities of the mock's tune + val entities
    test_scored.parquet   stage-2 probabilities of every kept test pair, with its country
check: v122's rule.json on test_scored ─► byte-identical to submissions/v122's matching file?
                                           no: stop
each false-merge weight w in 1.45 (v122), 2, 3, 4, 6:
    v107's rule choice on the mock's tune entities, tuned for 1 − L_FN − w·L_FP
        (expected-F0.5 decoding or the threshold grid, the higher tune score)
    ─► mock val: F0.5, est_public (at ×1.45), precision, recall, error counts
    ─► test: pairs by country, pairs removed / added vs v122
    w = 3 ─► submissions/v123_fp3/ + output/        w = 6 ─► submissions/v123_fp6/
tracking.log_result: the w = 3 arm as the headline, group E2, decision INVESTIGATE
```

## 1. Hypothesis

* **Change vs parent (v122):** the false-merge weight `w` of the score the rule is tuned for,
  1 − L_FN − w·L_FP on the mock's tune entities: 1.45 in v122 (`mock.FP_WEIGHT`), 3 in the
  upload candidate, with 2, 4 and 6 around it. Blocking, stage 1, stage 2, their probabilities
  and the candidate file are v122's.
* **Why (measured on the leaderboard):**

  | Upload | Change | est_public | Public F0.5 |
  |---|---|---|---|
  | v001 | first pipeline (LightGBM, 47 features) | – | 0.954 |
  | v101 | + core-name frequency features | – | 0.955 |
  | v103 | v101's matcher, rule re-tuned on the mock | 0.9610 | 0.961 |
  | v107 | two-stage matcher + rule tuned on the tight mock | 0.9659 | 0.966 |
  | v110 | + M3's feature groups, denser blocking | 0.9731 | 0.966 |
  | v121 | v110 with v107's French rows (diagnostic) | – | 0.965 |
  | 27 Sep 08:52 | assumed v122 (not yet logged in `LEADERBOARD.md`) | 0.9747 | 0.966 |

  The changes that bought precision moved the public score (v101 → v103 +0.006, v103 → v107
  +0.005). The recall-driven gains did not: v110 (mock pair recall 0.934 → 0.952) and v122
  (pair recall 0.952 → 0.955, est_public +0.0015 over v110) both left it at 0.966. Swapping
  France back to v107's rows (v121, 0.965) did not recover v110's predicted gain either.
* **Why the test should price false merges higher (v122's probabilities, §6):** candidates with
  0.1 < p < 0.9 per S1 are 0.153 on mock val, 0.193 on test India, 0.367 on test US and 0.802 on
  test France; the model's self-expected F0.5 of its own picks is 0.9921, 0.9908, 0.9852 and
  0.9644. France (15 % of test S1) is unseen in train. More uncertain candidates mean more wrong
  picks, and a rule tuned where wrong picks are rarer takes too many of them.
* **What it costs on the mock (measured, §5):** weights 1.45 / 2 / 3 / 4 / 6 give mock F0.5
  0.98306 / 0.98281 / 0.98235 / 0.98143 / 0.97938. Weight 3 drops a third of the false-merge
  pairs (2,827 → 1,904) and of the singleton merges (326 → 216) for 5,808 more missed pairs
  (51,412 → 57,220): −0.00071 mock F0.5.
* **Checks:** v122's rule applied to `test_scored.parquet` must rebuild v122's uploaded matching
  file byte for byte, or the script stops (§4.1); the 1.45 arm must re-tune v122's rule and
  reproduce its mock scores and error counts (§7).
* **Decision:** the mock cannot decide (its scores are the ×1 and ×1.45 ones), so the status is
  **INVESTIGATE, judged by the public upload**. Weight 3 (`submissions/v123_fp3/`, copied to
  `output/`) is uploaded first; if the public score rises, weight 6 (`submissions/v123_fp6/`) is
  next; if it falls, v122 stays.
* **Caveat:** the public score has 3 decimals and covers part of test; a change under 0.001 does
  not show, and the private part decides the final ranking.

## 2. Setup

Only `entity_resolution.config` is imported, for the paths: nothing here loads data or a model.
`read_json` returns None for a file the script has not written yet (or is writing), and
`finished()` guards every cell that needs the logged result. `log_entries` splits the run log
into its timestamped entries (`run_v123.log()` prints `[HH:MM:SS +  N.N min] message`; the
checker's output or a traceback can continue on the lines below), so a cell can quote what the
script logged about one step while the run is still going. `metrics.json` keys the arms by their
weight as a string (`"1.45"`, `"2.0"`, `"3.0"`, `"4.0"`, `"6.0"`); `BASE` is v122's weight and
`UPLOAD` the upload candidate's, and the tables label the arms `x1.45`, `x2`, ...

In [ ]:
import json

import pandas as pd
from IPython.display import display

from entity_resolution import config as C

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 30)

VERSION = "v123_tight_rule"
EXP_DIR = C.EXPERIMENTS / VERSION
PARENT_DIR = C.EXPERIMENTS / "v122_full_stack"
RUN_LOG = EXP_DIR / "run.log"                   # stdout + stderr of run_v123.py
BASE = "1.45"      # v122's false-merge weight (mock.FP_WEIGHT): test changes are counted from it
UPLOAD = "3.0"     # the arm the script copies to output/ (run_v123.UPLOAD)
KINDS = ["false_merge", "singleton_merge", "missed", "false_singleton"]
nan = float("nan")


def read_json(path) -> dict | None:
    """The JSON object in ``path``; None while the script has not written it (or is writing)."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None


def read_log() -> str | None:
    """The run log as text (UTF-8, or UTF-16 when a shell redirect wrote a BOM); None if absent."""
    if not RUN_LOG.exists():
        return None
    raw = RUN_LOG.read_bytes()
    utf16 = raw[:2] in (b"\xff\xfe", b"\xfe\xff")
    return raw.decode("utf-16" if utf16 else "utf-8", errors="replace")


def opens_entry(line: str) -> bool:
    """Whether ``line`` starts a log entry: run_v123.log() writes "[10:15:02 +  0.3 min] ..."."""
    return line[:1] == "[" and line[3:4] == ":" and " min] " in line[:24]


def log_entries(*needles: str) -> list[str]:
    """Run-log entries (a timestamped line and its continuation lines) holding any needle."""
    entries: list[list[str]] = []
    for line in (read_log() or "").splitlines():
        if opens_entry(line) or not entries:
            entries.append([line])
        else:                                   # the checker's output, a traceback
            entries[-1].append(line)
    texts = ["\n".join(lines).rstrip() for lines in entries]
    return [t for t in texts if any(n in t for n in needles)]


def finished() -> bool:
    """Whether the script has logged its result; prints "run not finished" while it has not."""
    if run is None:
        print("run not finished: run_v123.py writes metrics.json at its end "
              "(section 9 shows the run log so far)")
    return run is not None


def label(weight: str) -> str:
    """Table label of an arm: its false-merge weight as "x1.45", "x3"."""
    return f"x{float(weight):g}"


def rule_text(rule: dict) -> str:
    """A rule on one line: its family and parameters, as ``decision.rule_to_json`` wrote them."""
    params = [f"{k} {v}" for k, v in rule.items() if k not in ("kind", "one_to_one")]
    params.append("1-to-1" if rule.get("one_to_one") else "not 1-to-1")
    return f"{rule.get('kind', '?')}: " + ", ".join(params)


run = read_json(EXP_DIR / "metrics.json")       # the experiments.csv row + "metrics"
parent = read_json(PARENT_DIR / "metrics.json")
v123 = run["metrics"] if run else {}
v122 = parent["metrics"] if parent else {}
arms = v123.get("arms", {})                     # weight string -> that arm's record
print(f"v122 metrics.json read: {parent is not None}; run finished: {run is not None}"
      + (f"; decision {run.get('decision')}" if run else ""))
print(f"v122: rule {v122.get('rule')}, mock F0.5 {v122.get('f_beta', nan):.5f}, "
      f"est_public {v122.get('est_public', nan):.5f}")

### Parameters of this version

The script's constants (`run_v123.py`); `metrics.json` records the weights and the upload
weight, which the cell prints with the logged hypothesis:

* `WEIGHTS = (FP_WEIGHT, 2.0, 3.0, 4.0, 6.0)`: false-merge cost over missed-match cost, with
  `mock.FP_WEIGHT` = 1.45 (v122's);
* the expected-F0.5 grid: `GAMMAS` from 0.7 to 3.0 in steps of 0.1 and
  `MISSES = (0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.6)`; the threshold grid is v122's (`cfg.grid` of
  `run_v122.configs()`);
* `WRITE = {3.0: "v123_fp3", 6.0: "v123_fp6"}` (the arms written to `submissions/`) and
  `UPLOAD = 3.0` (the arm copied to `output/` and logged as the headline).

In [ ]:
if finished():
    print("weights:", v123.get("weights"), "| upload weight:", v123.get("upload_weight"))
    print("stage 1 + stage 2:", v123.get("stage1_stage2"))
    print("hypothesis:", v123.get("hypothesis"))

## 3. Data

The mock fold is rebuilt exactly as in v122 (`build_mock` on the train and val folds, with the
inner split's tune entities, v122's fit-side sample and `target_shape()`: per country the test's
pool size and pool records per S1). The rules are tuned on its tune entities and scored on its
val entities (340,084 S1: India 176,522, US 163,562). No feature is computed and no model is
trained; the probabilities are v122's, read from its artifacts:

* `mock_scored.parquet`: v122's stage-2 probabilities (the mean of seeds 42, 43 and 44; tune
  entities scored out of fold) of the kept pairs of the mock's tune and val entities;
* `test_scored.parquet`: v122's stage-2 probabilities of every kept test pair with its S1's
  country (France, India, US), written from v122's cached stage-1 outputs. Its pairs are the
  candidate file, v122's: the kept pairs do not depend on the rule.

The cell quotes what the script logged once both were loaded.

In [ ]:
entries = log_entries("mock built")
print("\n".join(entries) if entries else "nothing logged about the data yet")

## 4. Method

### 4.1 v122's probabilities, and the proof that they are v122's

v123 means something only if it decodes the probabilities v122 uploaded. Before the sweep the
script loads v122's chosen rule (`artifacts/two_stage/rule.json`: expected-F0.5 decoding, γ 1.5,
expected misses 0.2, at most 11 matches, 1-to-1), applies it to `test_scored.parquet` one country
at a time as `run_test_two_stage` does (`apply_rule`, then `sort_matches`), writes the files to a
temporary folder and compares the matching file with `submissions/v122/matching_results.tsv`
byte for byte (`filecmp.cmp(shallow=False)`). A mismatch stops the run before any arm is scored.

In [ ]:
entries = log_entries("rebuilds submissions/v122")
print("\n".join(entries) if entries else "the reproduction check has not run yet")
if finished():
    print(f"metrics.json: reproduces_v122 = {v123.get('reproduces_v122')}")

### 4.2 The rule at each false-merge weight

For each weight w, v107's rule choice on the mock's tune entities, as in v110 and v122, but for
the score 1 − L_FN − w·L_FP: each pool record stays only with its highest-probability S1
(1-to-1 across every present entity), then

* expected-F0.5 decoding (`tune_expected`: pair probability p^γ, m expected misses, over the
  `GAMMAS` × `MISSES` grid) and
* the threshold grid (`tune_mock`, v122's grid)

are tuned, and the family with the higher tune score gives the rule. A higher w makes a wrong
pick dearer against a missed one, so the tuned rule predicts fewer, surer pairs. The rules are
scored on the val entities (§5) and applied to the test pairs (§4.3). The 1.45 arm re-tunes
v122's own rule; the cell checks that it equals the rule v122 logged.

In [ ]:
if finished():
    display(pd.DataFrame({label(w): pd.Series(a.get("rule", {}), dtype=object)
                          for w, a in arms.items()}))
    retuned = {k: v for k, v in arms.get(BASE, {}).get("rule", {}).items() if k != "kind"}
    print(f"weight {BASE} re-tunes v122's logged rule {v122.get('rule')}: "
          f"{retuned == v122.get('rule')}")

### 4.3 Test files

Each arm's rule is applied to the test pairs country by country, as in §4.1; the script counts
the matches by country and against the 1.45 arm's (pairs removed and added), which is v122's own
decoding once the checks of §4.1 and §4.2 hold. Only weights 3 and 6 are written: `write_pairs`
with every test S1 as a row (an S1 without a match gets an empty list) and v122's kept pairs as
the candidate file, into `submissions/v123_fp3/` and `submissions/v123_fp6/`, each run through
our checker (`python -m entity_resolution.submission --check-ids`). Weight 3, the upload
candidate, is copied to `output/`. The candidate file is the same for every arm, so
`candidate_pairs.tsv` is still exactly the set the matcher scored. §9 shows what the script
logged about the files.

## 5. Evaluation on the mock

Scores on the mock's val entities (340,084 S1). Mock F0.5 (false merges ×1) and est_public
(the tight score at ×1.45, minus 0.0072) are measured at the mock's weights, not at the arm's,
so every arm above 1.45 is expected to score lower on both: the tables show the price of each
weight in mock terms, and the public upload judges whether the test pays it back. Error counts
are pairs (`evaluate.error_samples`, as in v110 and v122): `false_merge` a predicted pair that is
not true, of an entity with true matches; `singleton_merge` a predicted pair of a true
singleton; `missed` a true pair not predicted while its entity predicted something;
`false_singleton` the true pairs of an entity predicted empty (not in `missed`). Test removed /
added count each arm's test pairs against the 1.45 arm's, i.e. v122's.

Measured in the sweep before this run (the script logs the same numbers):

| Weight | Mock F0.5 | False merges | Singleton merges | Missed |
|---|---|---|---|---|
| 1.45 (v122) | 0.98306 | 2,827 | 326 | 51,412 |
| 2 | 0.98281 | 2,335 | 258 | 53,956 |
| 3 (upload) | 0.98235 | 1,904 | 216 | 57,220 |
| 4 | 0.98143 | 1,847 | 112 | 57,321 |
| 6 | 0.97938 | 1,147 | 93 | 67,036 |

### 5.1 The arms

In [ ]:
SCORES = {"mock F0.5": "f_beta", "est_public (x1.45)": "est_public",
          "pair precision": "pair_precision", "pair recall": "pair_recall"}
CHANGES = {"test pairs": "test_pairs", "test removed vs v122": "test_pairs_removed_vs_v122",
           "test added vs v122": "test_pairs_added_vs_v122"}


def arm_row(arm: dict) -> dict:
    """One arm's line: its rule, mock val scores and errors, and its test pairs against v122."""
    errors = arm.get("errors_mock", {})
    return {"rule": rule_text(arm.get("rule", {})),
            **{name: arm.get(key) for name, key in SCORES.items()},
            **{kind: errors.get(kind) for kind in KINDS},
            **{name: arm.get(key) for name, key in CHANGES.items()}}


if finished():
    sweep = pd.DataFrame({label(w): arm_row(a) for w, a in arms.items()}).T
    sweep[list(SCORES)] = sweep[list(SCORES)].astype(float).round(5)
    counts = KINDS + list(CHANGES)
    sweep[counts] = sweep[counts].astype("Int64")      # pair counts (<NA> where not logged)
    sweep.index.name = "false-merge weight"
    display(sweep)

### 5.2 Against v122

Each arm minus the 1.45 arm, and the trade it makes: the true pairs it gives up (`missed` +
`false_singleton`) per wrong pair it no longer predicts (`false_merge` + `singleton_merge`).
For scale, F0.5 counts a wrong pair four times a missed one in its denominator (β² = 0.25),
and the tight score ×1.45 on top of that. The `missed` column alone does not explain the F0.5
steps: an entity pushed to an empty prediction moves its pairs to `false_singleton`.

In [ ]:
if finished():
    base = arms.get(BASE, {})
    rows = {}
    for w, a in arms.items():
        err, err0 = a.get("errors_mock", {}), base.get("errors_mock", {})
        d = {kind: err.get(kind, nan) - err0.get(kind, nan) for kind in KINDS}
        dropped = -(d["false_merge"] + d["singleton_merge"])   # wrong pairs no longer predicted
        lost = d["missed"] + d["false_singleton"]              # true pairs given up for them
        rows[label(w)] = {**{f"d {name}": a.get(key, nan) - base.get(key, nan)
                             for name, key in SCORES.items()},
                          **{f"d {kind}": v for kind, v in d.items()},
                          "true pairs lost per wrong pair dropped":
                              lost / dropped if dropped > 0 else nan}
    deltas = pd.DataFrame(rows).T.astype(float).round(5)
    display(deltas.astype({f"d {kind}": "Int64" for kind in KINDS}))

### 5.3 By country and by entity type

Mock F0.5 per country (the mock has India and the US only), on the true singletons and on the
entities with matches, and the tight score at ×1.45 (est_public + 0.0072), then each arm's change
against the 1.45 arm.

In [ ]:
if finished():
    countries = sorted({k.removeprefix("mock_") for a in arms.values() for k in a
                        if k.startswith("mock_")})
    cols = {**{f"mock F0.5 {c}": f"mock_{c}" for c in countries},
            "F0.5 singletons": "f_beta_singletons", "F0.5 matched": "f_beta_matched",
            "f_tight (x1.45)": "f_tight"}
    parts = pd.DataFrame({label(w): {name: a.get(key) for name, key in cols.items()}
                          for w, a in arms.items()}).T.astype(float)
    display(parts.round(5))
    if label(BASE) in parts.index:
        print(f"change against {label(BASE)} (v122's rule):")
        display((parts - parts.loc[label(BASE)]).round(5))

## 6. Test-side evidence

The mock has no French entity and, from v110 on, stopped predicting the public score, so the
case for a costlier false merge rests on the test side. v122's own probabilities measure, without
any label, how hard each set is (measured on v122's artifacts before this run; the script records
the numbers in `evidence_test_harder`):

* **uncertain candidates per S1** (0.1 < p < 0.9): mock val 0.153, test India 0.193, US 0.367,
  France 0.802, i.e. 1.3×, 2.4× and 5.2× the mock's;
* **self-expected F0.5** of the picks, taking each candidate as true with its own probability
  (it ignores the true pairs blocking never found): mock val 0.9921, India 0.9908, US 0.9852,
  France 0.9644. The loss the model expects of itself (1 − F0.5) is 1.2× the mock's in India,
  1.9× in the US and 4.5× in France.

France holds 15 % of test S1 (259,452 of 1,732,544) and no training entity. The rule is one for
every country, so if the test is harder where the mock cannot look, a tighter rule should remove
the most pairs there: the second table shows each arm's test pairs by country against v122's,
the third how many test S1 keep a match.

In [ ]:
TEST_S1 = 1_732_544   # rows of v122's matching file: France 259,452, India 809,986, US 663,106
EVIDENCE = {"uncertain_cands_per_s1": "uncertain candidates per S1 (0.1 < p < 0.9)",
            "self_expected_f05": "self-expected F0.5 of the picks"}
if finished():
    table = pd.DataFrame(v123.get("evidence_test_harder", {})).T.astype(float)  # measure x set
    if "self_expected_f05" in table.index:              # the loss the model expects of itself
        table.loc["expected loss (1 - self-expected F0.5)"] = 1 - table.loc["self_expected_f05"]
    if "mock_val" in table.columns:                     # each test country against the mock
        for col in [c for c in table.columns if c != "mock_val"]:
            table[f"{col} / mock_val"] = table[col] / table["mock_val"]
    display(table.rename(index=EVIDENCE).round(4))

    by_country = pd.DataFrame({label(w): a.get("test_pairs_by_country", {})
                               for w, a in arms.items()}).astype(float)
    display(by_country.astype("Int64"))
    if label(BASE) in by_country.columns:
        print(f"test pairs by country, % change against {label(BASE)} (v122):")
        display((by_country.div(by_country[label(BASE)], axis=0) - 1).mul(100).round(2))
    matched = pd.Series({label(w): a.get("test_s1_matched") for w, a in arms.items()},
                        dtype=float)
    display(pd.DataFrame({"test S1 matched": matched.astype("Int64"),
                          "share of test S1": (matched / TEST_S1).round(4)}))

## 7. Log the result

The script logged the result itself, at its end (`tracking.log_result`: `metrics.json` in this
folder and the `v123` row of `experiments/experiments.csv`, stamped with the commit of `src/`),
with group `E2`, owner `M3`, parent `v122` and decision `INVESTIGATE`: the mock cannot rank the
arms against v122 (§5), so the public upload judges v123. The headline numbers (`mock_f05`,
`rule`, `f_beta`, `est_public`, ...) are the upload arm's (weight 3), next to v122's
(`v122_f_beta`, `v122_est_public`). Nothing is logged from this notebook: the cell shows the
logged row and checks that the headline is the weight-3 arm and that the 1.45 arm is v122 (its
rule, mock scores and error counts equal v122's logged ones). After the upload,
`make public V=v123 SCORE=0.xxxx` records the public score.

In [ ]:
def agree(logged, expected) -> bool:
    """Whether a logged value equals the expected one (numbers to 1e-9); False if either is None."""
    if logged is None or expected is None:
        return False
    if isinstance(logged, dict | list) or isinstance(expected, dict | list):
        return logged == expected
    return abs(float(logged) - float(expected)) < 1e-9


if finished():
    row = {k: v for k, v in run.items() if k != "metrics"}
    display(pd.Series(row, name="experiments.csv row", dtype=object).to_frame())
    chosen, base = arms.get(UPLOAD, {}), arms.get(BASE, {})
    retuned = {k: v for k, v in base.get("rule", {}).items() if k != "kind"}
    checks = {  # check: (logged, what it must equal)
        "upload weight": (v123.get("upload_weight"), float(UPLOAD)),
        "headline rule = x3 arm's": (v123.get("rule"), chosen.get("rule")),
        "headline mock F0.5 = x3 arm's": (v123.get("f_beta"), chosen.get("f_beta")),
        "csv mock_f05 = x3 arm's (4 d.p.)": (float(row.get("mock_f05") or nan),
                                             float(f"{chosen.get('f_beta', nan):.4f}")),
        "x1.45 rule = v122's": (retuned, v122.get("rule")),
        "x1.45 mock F0.5 = v122's": (base.get("f_beta"), v122.get("f_beta")),
        "x1.45 est_public = v122's": (base.get("est_public"), v122.get("est_public")),
        "x1.45 errors = v122's": (base.get("errors_mock"), v122.get("errors_mock")),
        "v122_est_public = v122's": (v123.get("v122_est_public"), v122.get("est_public")),
        "reproduces v122's matching file": (v123.get("reproduces_v122"), True),
    }
    display(pd.DataFrame({"logged": {k: str(a) for k, (a, _) in checks.items()},
                          "expected": {k: str(b) for k, (_, b) in checks.items()},
                          "agree": {k: agree(a, b) for k, (a, b) in checks.items()}}))
    print(f"group {row.get('group')!r}, owner {row.get('owner')!r}, parent "
          f"{row.get('parent')!r}, decision {row.get('decision')!r} "
          "(expected 'E2', 'M3', 'v122', 'INVESTIGATE')")

## 8. Conclusion

(filled in after the public score)

## 9. Test files

The script writes weights 3 and 6 (§4.3) to `submissions/v123_fp3/` and `submissions/v123_fp6/`,
runs our checker (`--check-ids`) on each (the `wrote ...: checker ...` lines) and copies weight 3
to `output/` (the `output/ now holds ...` line). The organisers' validator is not on this
machine: M1 runs it before an upload, which is logged in `LEADERBOARD.md` first.

Upload plan:

1. weight 3: `submissions/v123_fp3/`, the files now in `output/`;
2. if its public score rises, weight 6 (`submissions/v123_fp6/`) next;
3. if it falls, v122 stays.

The last 40 lines of the run log follow; while the run is going, this cell is the progress view.

In [ ]:
text = read_log()
if not text or not text.strip():
    print("run.log is missing or empty: the script has printed nothing yet")
else:
    if "does not reproduce v122" in text:
        state = "stopped: test_scored.parquet does not rebuild v122's matching file"
    elif log_entries("logged {"):
        state = "finished: result logged, test files written"
    elif log_entries("output/ now holds"):
        state = "test files written; logging the result"
    elif log_entries("] weight "):
        state = "sweeping the weights (or stopped: a traceback would end the tail)"
    else:
        state = "starting (or stopped: a traceback would end the tail)"
    print(f"run state: {state}\n" + "-" * 100)
    print("\n".join(text.rstrip().splitlines()[-40:]))